In [ ]:
from sklearn.decomposition import PCA
import torch
from datasets.mnist import DatasetMNISTBackground
import plotly.express as px
from misc.helpers import exp_name, repo_dir
from models.model_loader import load_model_from_folder
from tqdm import tqdm
from sklearn.manifold import TSNE, Isomap, LocallyLinearEmbedding
import umap

from models.vaes import VariationalLayer

mnist_ds = DatasetMNISTBackground(normalization="none", batch_size=200, output_label="both")
# mnist_ds = DatasetMNISTBackground(normalization="none", batch_size=200, output_label="both")
mnist_ds.preload_dataset(train=False)
labels_bg = [str(sample[1][0].item()) for sample in mnist_ds.wrapped_child_datasets["test"].items]
labels_digit = [str(sample[1][1].item()) for sample in mnist_ds.wrapped_child_datasets["test"].items]

def plot(encoder=None, device="cuda", outputs_are_images=True):
    if encoder == None:
        images = torch.vstack([sample[0] for sample in mnist_ds.wrapped_child_datasets["test"].items])
    else:
        encoder.to(device)
        images = []
        with torch.no_grad():
            for img, _ in tqdm(mnist_ds.get_dataloader(train=False)):
                img = img.to(device)
                images.extend(encoder(img))
        if outputs_are_images: 
            images = torch.cat(images, 0)
        else:
            images = torch.vstack(images)

    if outputs_are_images:
        images = images.flatten(1)
    images = images.cpu()

    # image_features = PCA(n_components=2).fit_transform(images)
    image_features = TSNE(n_components=2).fit_transform(images)
    # image_features = LocallyLinearEmbedding().fit_transform(images)
    # image_features = Isomap().fit_transform(images)
    # image_features = umap.UMAP().fit_transform(images)

    return image_features

[INF] 18-09_12:17 - DatasetMNISTBackground - Preloading the test dataset (no encoding).
[LOG] 18-09_12:17 - DatasetMNISTBackground - Creating new test dataset.


100%|██████████| 10000/10000 [00:00<00:00, 30477.87it/s]


In [4]:
from models.model_helpers import SetToZero
from plotly.subplots import make_subplots

# default_vae_model = load_model_from_folder(repo_dir("experiments", exp_name("00_0"), "results", "1e"), model_name="model", weights_file_name=f"model_200")
split_vae_model = load_model_from_folder(repo_dir("experiments", exp_name("00_0"), "results", "2e"), model_name="model", weights_file_name=f"model_300")

features_base = plot(None)
features_split = plot(torch.nn.Sequential(split_vae_model.encoder, VariationalLayer(return_latent_only=True)), outputs_are_images=False)
features_merged = plot(torch.nn.Sequential(split_vae_model.encoder, VariationalLayer(return_latent_only=True), SetToZero(0)), outputs_are_images=False)


[LOG] 18-09_12:18 - DatasetMNISTBackground - Using existing test dataloader.


100%|██████████| 50/50 [00:00<00:00, 1056.58it/s]


[LOG] 18-09_12:18 - DatasetMNISTBackground - Using existing test dataloader.


100%|██████████| 50/50 [00:00<00:00, 1068.27it/s]


In [6]:
fig = make_subplots(rows=2, cols=3, shared_xaxes=True, shared_yaxes=True, horizontal_spacing=.01, vertical_spacing=.01)
def add(px_plot, col, row):
    for trace in px_plot["data"]:
        fig.append_trace(trace, row=row, col=col)

add(px.scatter(x=features_base[:, 0], y=features_base[:, 1], color=labels_bg, opacity=.4), row=1, col=1)
add(px.scatter(x=features_base[:, 0], y=features_base[:, 1], color=labels_digit, opacity=.4), row=2, col=1)

add(px.scatter(x=features_split[:, 0], y=features_split[:, 1], color=labels_bg, opacity=.4), row=1, col=2)
add(px.scatter(x=features_split[:, 0], y=features_split[:, 1], color=labels_digit, opacity=.4), row=2, col=2)

add(px.scatter(x=features_merged[:, 0], y=features_merged[:, 1], color=labels_bg, opacity=.4), row=1, col=3)
add(px.scatter(x=features_merged[:, 0], y=features_merged[:, 1], color=labels_digit, opacity=.4), row=2, col=3)

fig.update_yaxes(scaleanchor="x", scaleratio=1)
scale = 600
fig.update_layout(height=2*scale, width=3*scale, margin={"l": 0, "r": 0, "t": 0, "b": 0}, showlegend=False)
fig